# 14. File I/O & Serialization: Beginner Guide

### 📌 Overview
Master **14. File I/O & Serialization: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Binary Serialization (`.npy`)**: Covers `np.save()` and `np.load()`.
- **Multi-Array Archives (`.npz`)**: Covers `np.savez()` and `np.savez_compressed()`.
- **Memory-Mapped Files (`np.memmap`)**: Out-of-core virtual memory mapping for arrays larger than RAM.
- **Text Parsing**: Covers `np.loadtxt()` and `np.genfromtxt()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Binary Persistence: `np.save()` & `np.load()`
- **What it does:** Serializes transaction amounts to fast `.npy` binary format.
- **Syntax:** `np.save('scratch/tx_amounts.npy', amounts)`
- **Operation:** `os.makedirs('scratch', exist_ok=True)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

In [2]:
os.makedirs('scratch', exist_ok=True)
np.save('scratch/tx_amounts.npy', amounts)
loaded_amounts = np.load('scratch/tx_amounts.npy')
print('Saved & Loaded array equal?:', np.array_equal(amounts, loaded_amounts))

Saved & Loaded array equal?: True


### 🔹 Uncompressed Archive: `np.savez()`
- **What it does:** Bundles amounts, fraud flags, and account ages into a single `.npz` archive.
- **Syntax:** `np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

In [3]:
np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags, ages=account_ages)
with np.load('scratch/transactions_all.npz') as arch:
    print('Archived Keys:', arch.files)
    print('Loaded fraud array shape:', arch['fraud'].shape)

Archived Keys: ['amounts', 'fraud', 'ages']
Loaded fraud array shape: (14251,)


### 🔹 Compressed Multi-Array Archive: `np.savez_compressed()`
- **What it does:** Applies zip compression saving 70% disk space.
- **Syntax:** `np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

In [4]:
np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)
print('Compressed npz written. Size:', os.path.getsize('scratch/tx_compressed.npz'), 'bytes')

Compressed npz written. Size: 48423 bytes


### 🔹 Out-of-Core Memory Mapping: `np.memmap()`
- **What it does:** Memory-maps binary transaction records directly into virtual RAM without loading file into heap.
- **Syntax:** `np.memmap('scratch/large_tx.dat', dtype='float64', mode='w+', shape=amounts.shape)`
- **Key Note:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.

In [5]:
mmap_tx = np.memmap('scratch/mmap_transactions.dat', dtype='float64', mode='w+', shape=amounts.shape)
mmap_tx[:] = amounts[:]
mmap_tx.flush()
print('Memory-mapped transaction buffer created. Shape:', mmap_tx.shape)

Memory-mapped transaction buffer created. Shape: (14251,)


### 🔹 Text Parsing: `np.loadtxt()` & `np.genfromtxt()`
- **What it does:** Parses comma-delimited numeric columns directly from CSV.
- **Syntax:** `np.genfromtxt('data/raw_transactions.csv', delimiter=',', skip_header=1, usecols=(3, 7))`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

In [6]:
tx_numeric_cols = np.genfromtxt(csv_path, delimiter=',', skip_header=1, usecols=(3, 7), max_rows=5)
print('Parsed Numeric Columns (amount, account_age):\n', tx_numeric_cols)

Parsed Numeric Columns (amount, account_age):
 [[ 607.78    8.  ]
 [1819.11   28.  ]
 [  64.08   91.  ]
 [1025.73   50.  ]
 [ 772.74    5.  ]]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Streaming Chunked Aggregation on Memmap
- **Objective:** Q1: Streaming Chunked Aggregation on Memmap
- **Approach:** Calculate total transaction spend across memory-mapped file in chunks.
- **Syntax:** `mmap_tx[start:end].sum()`

In [7]:
chunk_sz = 2500
total_mmap_sum = sum(mmap_tx[i:i+chunk_sz].sum() for i in range(0, len(mmap_tx), chunk_sz))
print(f'Total Spend across Memmap Chunks: ${total_mmap_sum:,.2f}')

Total Spend across Memmap Chunks: $14,326,935.50
